# MLP from scratch for MNIST classification 

## Introduction
In this notebook, we will code from scratch an MLP (mutli-layer perceptron) for the classification task on MNIST dataset. Everything in the algorithm -from forward propagation to backpropagation and optimization- were be written from scratch.
We will also experiment with different configurations and architectures to observe how various parameters affect the learning process. 

In [405]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [406]:
# Importing the training and the validation datesets
trainset = pd.read_csv("digit-recognizer/train.csv")
testset = pd.read_csv("digit-recognizer/test.csv")

In [407]:
print("trainset shape :",trainset.shape)
print("validset shape :",testset.shape)

trainset shape : (42000, 785)
validset shape : (28000, 784)


In [394]:
trainset.columns

Index(['label', 'pixel0', 'pixel1', 'pixel2', 'pixel3', 'pixel4', 'pixel5',
       'pixel6', 'pixel7', 'pixel8',
       ...
       'pixel774', 'pixel775', 'pixel776', 'pixel777', 'pixel778', 'pixel779',
       'pixel780', 'pixel781', 'pixel782', 'pixel783'],
      dtype='object', length=785)

In [408]:
testset.columns

Index(['pixel0', 'pixel1', 'pixel2', 'pixel3', 'pixel4', 'pixel5', 'pixel6',
       'pixel7', 'pixel8', 'pixel9',
       ...
       'pixel774', 'pixel775', 'pixel776', 'pixel777', 'pixel778', 'pixel779',
       'pixel780', 'pixel781', 'pixel782', 'pixel783'],
      dtype='object', length=784)

In [ ]:
print(trainset.isnull().sum().sum())
print(testset.isnull().sum().sum())

# there are no null values in the dataset

0
0
(26880, 785)
(6720, 785)


In [ ]:
#splitting the training dataset into the training and the validation datasets
validset = trainset.sample(frac=0.2, random_state=42)
trainset = trainset.drop(validset.index)

print(trainset.shape)
print(validset.shape)

In [395]:
# N = 100 #number of points per class
# D = 2
# K = 3
# X = np.zeros((N*K, D)) #data matrix (each row = single example)
# print(X.shape)
# y = np.zeros(N*K, dtype='uint8') #class labels
# print(y.shape)
# for j in range(K) : 
#     ix = range(N*j, N*(j+1))
#     r = np.linspace(0.0, 1, N) #radius
#     t = np.linspace(j*4, (j+1)*4, N) + np.random.randn(N)*0.2 #theta
#     X[ix] = np.c_[r*np.sin(t), r*np.cos(t)]
#     y[ix] = j

# # let s visualize the data:
# plt.scatter(X[:, 0], X[:, 1], c=y, s=40, cmap=plt.cm.Spectral)
# plt.show()


In [ ]:
X = trainset.drop("label", axis=1).values
y = trainset["label"].values

## Building the model 
### Initializing the parameters
The literature suggests multiple methods of initialization. We choose among them the He initialization where the weights are drawn as : 
$$ W \sim \mathcal{N}(0, \sqrt{2/n_{\text{in}}}) $$

We decided to use He initialization because it is the one that suits the most Relu, the activation function that we will use in our MLP. 

How He initialization is adapted to the Relu function ?
Relu sets half of the inputs on average to zero.
So roughly 50% of activations vanish.
If we used a naive initialization, ,the variance of activations would shrink layer after layer, causing :
vanishing signals
slow training
He initialization compensates for this by doubling the variance.


In [396]:
def he_initial(n_in, n_out):
    W = np.random.randn(n_in, n_out) * np.sqrt(2. / n_in)
    b = np.zeros((1, n_out)) #bias is initialized to zero
    return W, b


### Fully connected layer
The fully connected layer is a layer where each neuron is connected to each input. This design allows the network to learn complex relationships between features. We implement it as a class so it can be easily reused and combined with other layers in our MLP.

In [397]:
class FCLayer :
    """fully connected layer"""

    def __init__(self, n_in, n_out):
        self.n_in = n_in
        self.n_out = n_out
        self.W, self.b = he_initial(n_in,n_out)


    def forward(self, X):
        return np.dot(X,self.W) + self.b

### Activation functions


The activation functions are very important to neural networks. Without them, stacking multiple linear layers is mathematically equivalent to a single linear layer, making the network incapable of representing non-linear functions.
are different activation functions used in different contexts. In our neural network, we used Relu for the first and second layer and softmax for the last one.

Why softmax ?
Softmax is used generally for classification tasks. It converts logits to probabilities that sum to 1, enabling probabilistic interpretation for mutually exclusive classes.

Why Relu ?
Relu is very popular in deep learning for differents reasons. The computation of the function and its derivative are light so it does not slow down the neither the forwardpropagation or backpropagation. Also, it does not cause the gradient to vanish. But Relu has its limitations : 
- Dying ReLU problem: Neurons can become permanently inactive (output 0 for all inputs)

- Not zero-centered: Can cause optimization issues. This quite tricky to explain. Actually, in the particular case where all the inputs x have the same sign, the gradient will also have the same sign, so the update of the different weights due to the gradient descent moves only in one direction(negative or positive). This leads to a slow convergence towards the minima because the algorithm zigzags instead of going directly.


In [398]:
def Relu(x) :
    return np.maximum(0,x)

def relu_derivative(x):
    return (x > 0).astype(float)

In [399]:
def softmax(x): 
    exp_scores = np.exp(x - np.max(x, axis=1, keepdims=True))  # for numerical stability
    return exp_scores / np.sum(exp_scores, axis=1, keepdims=True)


### Loss
We use the cross-entropy loss because this is a classification problem. To properly understand this loss function, it is first important to understand the role of the softmax function.

The purpose of the softmax function is to convert the raw scores produced by the network into a probability distribution over the different classes. From an information theory perspective, cross-entropy measures the discrepancy between two probability distributions: the true distribution, represented by the one-hot encoded labels, and the predicted distribution, given by the output of the softmax function. The objective is therefore to minimize the difference between these two distributions, which corresponds to minimizing the loss.

This loss can also be interpreted from a probabilistic perspective. Minimizing the cross-entropy loss is equivalent to maximizing the log-likelihood of the correct class (or equivalently minimizing the negative log-likelihood), which leads to the optimal model parameters.

In [400]:
def compute_loss(outputs, y, reg_lambda, W):
    one_hot_labels = np.zeros((y.shape[0], K))
    one_hot_labels[np.arange(len(y)),y] = 1
    data_loss  = -np.mean(np.sum(one_hot_labels * np.log(outputs), axis=1))
    reg_loss = 0
    for w in W : 
        reg_loss += 0.5 * reg_lambda * np.sum(w*w)
    return data_loss + reg_loss


### Backpropagation
Backpropagation is an algorithm for efficiently computing gradients in neural networks. In deep networks, the loss as a function of the weights is a highly complex composition of operations, making direct analytical derivation of the gradients impractical. Backpropagation leverages the chain rule to compute gradients layer by layer, starting from the output and moving backward toward the input. By storing intermediate values from the forward pass and reusing them during the backward pass, the algorithm avoids redundant computations and keeps the per-layer gradient calculations relatively simple. 

In [401]:
def backpropagation(X, Z1, A1, Z2, A2, Z3, A3, y, W2, W3):

    m = X.shape[0] #number of examples
    #output layer gradients
    one_hot_encoding = np.zeros((m,A3.shape[1]))
    one_hot_encoding[np.arange(m), y] = 1
    dZ3 = A3 - one_hot_encoding
    dW3 = np.dot(A2.T, dZ3)/m
    db3 = np.sum(dZ3, axis=0, keepdims=True) / m

    #second hidden layer gradients
    dA2 = np.dot(dZ3, W3.T) 
    dZ2 = dA2 * relu_derivative(Z2)
    dW2 = np.dot(A1.T, dZ2)/m
    db2 = np.sum(dZ2, axis=0, keepdims=True) / m

    # first hidden layer gradients
    dA1 = np.dot(dZ2, W2.T)
    dZ1 = dA1 * relu_derivative(Z1)
    dW1 = np.dot(X.T, dZ1) / m
    db1 = np.sum(dZ1, axis=0, keepdims=True) / m

    return  dW1, db1, dW2, db2, dW3, db3

### Optimization

In [402]:
def optimizer_GD(learning_rate, W, b,dW, db): 
    "optimizes the loss function using the gradient descent algorithm"
    for i in range(len(W)):
        W[i] -= learning_rate * dW[i]
        b[i] -= learning_rate * db[i]
    


In [403]:
n_in, n_hidden_1, n_hidden_2, n_out = 2, 200,20, 3
reg_lambda = 0.01
learning_rate = 0.01


layer1 = FCLayer(n_in, n_hidden_1)
layer2 = FCLayer(n_hidden_1, n_hidden_2)
layer3 = FCLayer(n_hidden_2, n_out)

epochs = 100

for epoch in range(epochs): 
    Z_1 = layer1.forward(X)
    A_1 = Relu(Z_1)

    Z_2 = layer2.forward(A_1)
    A_2 = Relu(Z_2)

    Z_3 = layer3.forward(A_2)
    A_3 = softmax(Z_3)


    loss = compute_loss(A_3, y, reg_lambda, W=[layer1.W, layer2.W, layer3.W])
    ## backpropagation code will go here
    dW1, db1, dW2, db2, dW3, db3 = backpropagation(X, Z_1, A_1, Z_2, A_2, Z_3, A_3,y, layer2.W, layer3.W)    

    ## update weights and biases using the optimizer
    optimizer_GD(learning_rate, W=[layer1.W, layer2.W, layer3.W], b=[layer1.b, layer2.b, layer3.b], dW=[dW1, dW2, dW3], db=[db1, db2, db3])

    # if epoch % 10 == 0 : 
    #     print(f"epoch n : {epoch} / loss : {loss}")

print(A_3.shape)
predicted_class = np.argmax(A_3, axis=1)
print(A_3[:5])
print(predicted_class[:5])
print(np.mean(predicted_class==y))

(300, 3)
[[0.3320813  0.3237725  0.34414621]
 [0.33068777 0.3330999  0.33621233]
 [0.33143237 0.34123127 0.32733636]
 [0.32714208 0.34801147 0.32484645]
 [0.32909308 0.35425221 0.31665471]]
[2 2 1 1 1]
0.5066666666666667
